In [1]:
from sedona.spark import SedonaContext
from IPython.display import Image, display

from sedona.spark import dataframe_to_arrow
import geopandas as gpd
from sedona.spark.stats.weighting import add_distance_band_column
from sedona.spark.maps.SedonaKepler import SedonaKepler
import os

In [2]:
%%capture
bucket_name = os.environ.get("SEDONA_SOURCE_BUCKET", "apache-sedona-book")

config = SedonaContext.builder()

sedona = SedonaContext.create(config.getOrCreate())

sedona.sparkContext.setLogLevel("ERROR")

sc = sedona.sparkContext
sedona.sparkContext.setCheckpointDir("checkpoint")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/12 20:07:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/12 20:07:57 WARN UDTRegistration: Cannot register UDT for org.geotools.coverage.grid.GridCoverage2D, which is already registered.
25/09/12 20:07:57 WARN SimpleFunctionRegistry: The function rs_union_aggr replaced a previously registered function.
25/09/12 20:07:57 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.geom.Geometry, which is already registered.
25/09/12 20:07:57 WARN UDTRegistration: Cannot register UDT for org.apache.sedona.common.S2Geography.Geography, which is already registered.
25/09/12 20:07:57 WARN UDTRegistration: Cannot register UDT for org.locationtech.jts.index.SpatialIndex, which is already registered.
25/09/12 20:07:57 WARN SimpleFunctionRegistry: The function st_envelop

In [3]:
from pyspark.sql.functions import pandas_udf, udf
from pyspark.sql.types import FloatType
import pandas as pd
import pyspark.sql.functions as f

germany_polygon = """POLYGON((
  8.9765 47.2701,
  13.8396 47.2701,
  13.8396 50.5646,
  8.9765 50.5646,
  8.9765 47.2701
))""".replace("\n", "").replace("  ", " ")

connector = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/bavaria/connector")
transportation = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/bavaria/transportation").repartition(100)
infrastructure = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/infrastructure")
places = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/bavaria/places")
accidents = sedona.read.format("geoparquet").load(f"s3a://{bucket_name}/source_data/bavaria/accidents/")

In [4]:
values_mapping = {
    "accident_category": {
        "1": "fatal",
        "2": "major_injuries",
        "3": "minor_injuries"
    },
    "lighting_condition": {
        "0": "Daylight",
        "1": "Twilight",
        "2": "Darkness"
    }
}

In [5]:
h3_cells = sedona.sql(
    f"""
    WITH h3_cells AS (
        SELECT
            id,
            ST_H3ToGeom(ARRAY(id))[0] AS geom
        LATERAL VIEW EXPLODE(ST_H3CellIDs(ST_GeomFromText('{germany_polygon}'), 7, true)) AS id
    )
    SELECT 
        id,
        ST_Transform(geom, 'epsg:4326', 'epsg:3044') AS geometry
    FROM h3_cells
    """
)

In [8]:
SedonaKepler.create_map(accidents.selectExpr("ST_Transform(geometry, 'epsg:3044', 'epsg:4326')").limit(10000))

/usr/local/lib/python3.10/dist-packages/keplergl/keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string
                                                                                

KeplerGl(data={'unnamed': {'index': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20,…

In [20]:
# Create weight dataframe

In [6]:
from sedona.stats.hotspot_detection.getis_ord import g_local
import pyspark.sql.functions as f
from sedona.stats.weighting import add_distance_band_column

/tmp/ipykernel_319/3606826859.py:1: DeprecationWarning: Importing from 'sedona.stats.hotspot_detection.getis_ord' is deprecated. Please use 'sedona.spark.stats.hotspot_detection.getis_ord' instead.
  from sedona.stats.hotspot_detection.getis_ord import g_local
/tmp/ipykernel_319/3606826859.py:3: DeprecationWarning: Importing from 'sedona.stats.weighting' is deprecated. Please use 'sedona.spark.stats.weighting' instead.
  from sedona.stats.weighting import add_distance_band_column


In [7]:
accidents_agg = h3_cells.alias("h").\
    join(accidents.alias("a"), on=f.expr("ST_Intersects(h.geometry, a.geometry)")).\
    groupBy("h.id").\
    agg(f.count("*").alias("value"))

In [8]:
accidents_agg_values = h3_cells.join(accidents_agg.select("id", "value"), on="id", how="left").\
    selectExpr("id", "geometry", "Coalesce(value, 0) AS value").repartition(10)

In [9]:
weights_df = add_distance_band_column(
    dataframe=accidents_agg_values,
    threshold=200.0,
    include_self=True
)

In [10]:
from sedona.stats.hotspot_detection.getis_ord import g_local

gi_df = g_local(
    dataframe=weights_df,
    x="value",
    star=True
)

In [11]:
gi_df.cache().count()

27534

In [12]:
munich_area_wkt = """
POLYGON((
  10.438908 47.667611,
  12.663354 47.667611,
  12.663354 48.685899,
  10.438908 48.685899,
  10.438908 47.667611
))
""".replace("\n", "").replace(". ", " ")

df = gi_df.where(
    f"ST_Intersects(geometry, ST_Transform(ST_GeomFromText('{munich_area_wkt}'), 'epsg:4326', 'epsg:3044'))"
).select("id", "Z", "P", "geometry")

In [51]:
IPython.display.HTML(visualize_map(df))

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


# basic statistics

In [13]:
accidents_category_mapping_expr = f.create_map(
    [f.lit(x) for pair in values_mapping["accident_category"].items() for x in pair]
)

lighting_condition_category_mapping_expr = f.create_map(
    [f.lit(x) for pair in values_mapping["lighting_condition"].items() for x in pair]
)

aggregated_accidents = accidents.groupBy("accident_category", "lighting_condition").\
    count().\
    orderBy(f.col("accident_category")).\
    select(
        accidents_category_mapping_expr[f.col("accident_category")].alias("category"),
        lighting_condition_category_mapping_expr[f.col("lighting_condition")].alias("lighting_condition"),
        f.col("count")
    )

In [14]:
summarized = aggregated_accidents.groupBy("lighting_condition").sum().\
    selectExpr("lighting_condition", "`sum(count)` AS count")

In [15]:
aggregated_accidents.alias("a").join(summarized.alias("s"), on="lighting_condition").\
    selectExpr("category", "lighting_condition", "ROUND(a.count/s.count * 100, 2) AS percent").\
    orderBy(f.col("category"), f.desc("percent")).\
    show()

+--------------+------------------+-------+
|      category|lighting_condition|percent|
+--------------+------------------+-------+
|         fatal|          Darkness|   1.35|
|         fatal|          Twilight|   1.05|
|         fatal|          Daylight|   0.96|
|major_injuries|          Darkness|  17.28|
|major_injuries|          Daylight|  16.95|
|major_injuries|          Twilight|  16.73|
|minor_injuries|          Twilight|  82.22|
|minor_injuries|          Daylight|  82.09|
|minor_injuries|          Darkness|  81.37|
+--------------+------------------+-------+



# Feature engineering

In [16]:
from pyspark.sql.functions import pandas_udf, udf
from pyspark.sql.types import FloatType, BooleanType
import pandas as pd
from shapely.geometry.linestring import LineString
from sedona.spark.sql.functions import sedona_vectorized_udf
import math

german_speed_limits = {
    'motorway': 130,
    'trunk': 100,
    'primary': 80,
    'secondary': 80,
    'tertiary': 50,
    'residential': 30
}

typical_widths = {
    'motorway': 25,
    'trunk': 20,
    'primary': 12,
    'secondary': 10,
    'tertiary': 8,
    'residential': 6,
    'service': 5,
}

widths_mapping_expr = f.create_map(
    [f.lit(x) for pair in typical_widths.items() for x in pair]
)

@pandas_udf(FloatType())
def map_speed_limit(rss: pd.Series,rcs: pd.Series) -> pd.Series:
    def first_non_empty(road_speed, road_cls):
        default = german_speed_limits[road_cls]
        if road_speed is None:
            return default
            
        for element in road_speed:
            max_speed = element.get("max_speed")
            if not max_speed:
                continue
            speed_value = max_speed.get("value")
            
            if speed_value:
                return speed_value
                
        return default

    df = pd.DataFrame({'road_class': rcs, 'road_speed': rss})
        
    return df.apply(lambda x: first_non_empty(
        x["road_speed"], x["road_class"]
    ), axis=1)


def match_road_flag(flag_value: str):
    @pandas_udf(BooleanType())
    def is_road_flag(road_flags: pd.Series) -> pd.Series:
        def contains_road_flag(flags: list):
            if flags is None:
                return False

            for flag in flags:
                if flag_value in flag["values"]:
                    return True

            return False

        return road_flags.apply(lambda x: contains_road_flag(x))

    return is_road_flag


@sedona_vectorized_udf(return_type=FloatType())
def get_average_curvature(line: LineString):
    coords = list(line.coords)
    if len(coords) < 3:
        return 0

    from math import atan, pi
    
    def slope(p1, p2):
        dx = p2[0] - p1[0]
        dy = p2[1] - p1[1]
        if dx == 0.0:
            return None
        return dy / dx
    
    def angle_between(a, b, c):
        m1 = slope(a, b)
        m2 = slope(b, c)
    
        if m1 is None and m2 is None:
            return 0
        elif m1 is None or m2 is None:
            return pi / 2
    
        tan_theta = (m2 - m1) / (1 + m1 * m2)
        angle = atan(tan_theta)
    
        return abs(angle)

    angles = []
    for i in range(1, len(coords)-1):
        angle = angle_between(coords[i-1], coords[i], coords[i+1])
        angles.append(angle)

    return sum(angles) / len(angles)

In [17]:
crossings = infrastructure.where("class == 'crossing'").\
    selectExpr(
        "id",
        "ST_Transform(geometry, 'epsg:4326', 'epsg:3044') AS geometry"
    )

cycleways = transportation.where("class == 'cycleway'").\
    select("id", "geometry")

is_bridge = match_road_flag("is_bridge")(
    f.col("road_flags")
).alias("is_bridge")

is_tunnel = match_road_flag("is_tunnel")(
    f.col("road_flags")
).alias("is_tunnel")

is_under_construction = match_road_flag("is_under_constructruction")(
    f.col("road_flags")
).alias("is_under_construction")

road_classes = ('motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'residential')

enriched_transportation = (
    transportation
        .where(f"class in {road_classes}")
        .withColumn(
            "geometry",
            f.expr("ST_Transform(geometry, 'epsg:4326', 'epsg:3044')"))
        .withColumn(
            "width", 
            widths_mapping_expr[f.col("class")])
        .withColumn(
            "max_speed",
            map_speed_limit(f.col("speed_limits"), f.col("class")))
        .select(
            "id",
            "geometry",
            "width",
            "max_speed",
            is_bridge,
            is_tunnel,
            is_under_construction,
            get_average_curvature("geometry").alias("curvature"),
            "quality"
        )
)

In [18]:
roads_with_accidents = enriched_transportation.alias("t").\
    join(
        accidents.alias("a"),
        on=f.expr("ST_DWithin(t.geometry, a.geometry, t.width)")
    ).selectExpr("t.*", "a.lighting_condition", "a.accident_category", "a.promiles")

In [ ]:
roads_with_accidents.count()

290270

In [21]:
roads_with_accidents.cache().count()

290270

In [26]:
roads_feature_values = roads_with_accidents.alias("ra").\
    join(
        crossings.alias("c"),
        on=f.expr("ST_KNN(ra.geometry, c.geometry, 1)")
    ).\
    selectExpr(
        "ra.id",
        "ra.geometry",
        "ra.max_speed AS ms",
        "ra.is_bridge AS is_b",
        "ra.is_tunnel AS is_t",
        "ra.is_under_construction AS is_uc",
        "ROUND(ra.curvature, 3) AS c",
        "ra.lighting_condition AS lc",
        "ra.accident_category AS ac",
        "ROUND(ST_Distance(c.geometry, ra.geometry), 2) AS distance",
        "ra.promiles",
        "ra.quality"
    )

In [27]:
roads_feature_values.show()

These filters will be applied to the object side reader before the KNN join is executed. 
If you intend to apply the filters after the KNN join, please ensure that you materialize the KNN join results before applying the filters. 
For example, you can use the following approach:

Scala Example:
val knnResult = knnJoinDF.cache()
val filteredResult = knnResult.filter(condition)

SQL Example:
CREATE OR REPLACE TEMP VIEW knnResult AS
SELECT * FROM (
  -- Your KNN join SQL here
) AS knnView
CACHE TABLE knnResult;
SELECT * FROM knnResult WHERE condition;
These filters will be applied to the object side reader before the KNN join is executed. 
If you intend to apply the filters after the KNN join, please ensure that you materialize the KNN join results before applying the filters. 
For example, you can use the following approach:

Scala Example:
val knnResult = knnJoinDF.cache()
val filteredResult = knnResult.filter(condition)

SQL Example:
CREATE OR REPLACE TEMP VIEW knnResult AS
SELECT * FR

[Stage 73:==============================================>          (9 + 2) / 11]

+--------------------+--------------------+-----+-----+-----+-----+-----+---+---+--------+-----------+-------+
|                  id|            geometry|   ms| is_b| is_t|is_uc|    c| lc| ac|distance|   promiles|quality|
+--------------------+--------------------+-----+-----+-----+-----+-----+---+---+--------+-----------+-------+
|0891faa02e67ffff0...|LINESTRING (51566...| 50.0|false|false|false|0.243|  0|  2|   11.37|  2.0599341|      3|
|0891faa02e67ffff0...|LINESTRING (51566...| 50.0|false|false|false|0.243|  2|  3|   11.37| 0.26234674|      3|
|0891faa02e67ffff0...|LINESTRING (51566...| 50.0|false|false|false|0.243|  0|  3|   11.37|0.006066638|      3|
|0891faa02e67ffff0...|LINESTRING (51566...| 50.0|false|false|false|0.243|  0|  3|   11.37|0.007760236|      3|
|0861fa80efffffff0...|LINESTRING (57995...|120.0| true|false|false|0.018|  0|  2|  198.37| 0.16522339|      3|
|0861fa80efffffff0...|LINESTRING (57995...|120.0| true|false|false|0.018|  0|  2|  198.37|  0.7622792|      3|
|

In [28]:
roads_feature_values.\
    withColumn("distance_bucket", f.when(f.col("distance") <= 100, f.lit("<=100m")).otherwise(f.lit(">100m"))).\
    groupBy("distance_bucket", "ac").\
    count().\
    orderBy(f.col("distance_bucket"), f.col("ac")).\
    show()

These filters will be applied to the object side reader before the KNN join is executed. 
If you intend to apply the filters after the KNN join, please ensure that you materialize the KNN join results before applying the filters. 
For example, you can use the following approach:

Scala Example:
val knnResult = knnJoinDF.cache()
val filteredResult = knnResult.filter(condition)

SQL Example:
CREATE OR REPLACE TEMP VIEW knnResult AS
SELECT * FROM (
  -- Your KNN join SQL here
) AS knnView
CACHE TABLE knnResult;
SELECT * FROM knnResult WHERE condition;
These filters will be applied to the object side reader before the KNN join is executed. 
If you intend to apply the filters after the KNN join, please ensure that you materialize the KNN join results before applying the filters. 
For example, you can use the following approach:

Scala Example:
val knnResult = knnJoinDF.cache()
val filteredResult = knnResult.filter(condition)

SQL Example:
CREATE OR REPLACE TEMP VIEW knnResult AS
SELECT * FR

[Stage 83:===========================================>              (3 + 1) / 4]

+---------------+---+------+
|distance_bucket| ac| count|
+---------------+---+------+
|         <=100m|  1|   613|
|         <=100m|  2| 16283|
|         <=100m|  3|135919|
|          >100m|  1|  1699|
|          >100m|  2| 22715|
|          >100m|  3|113041|
+---------------+---+------+



In [29]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("id").orderBy(f.desc("count"))
similar_size_buckets = Window.partitionBy("ac").orderBy("id")

roads_feature_values_ranked = roads_feature_values.\
    groupBy("id", "ac", "lc", "promiles").\
    agg(
        f.col("id"),
        f.count("*").alias("count"),
        f.first("ms").alias("ms"),
        f.first("is_b").alias("is_b"),
        f.first("is_t").alias("is_t"),
        f.first("is_uc").alias("is_uc"),
        f.first("c").alias("c"),
        f.first("distance").alias("distance"),
        f.first("quality").alias("quality"),
    ).\
    withColumn("rank", row_number().over(window_spec)).\
    where("rank == 1").\
    withColumn("bucket_size", row_number().over(similar_size_buckets)).\
    where("bucket_size < 1000").\
    drop("rank", "bucket_size")

In [30]:
roads_feature_values_ranked.cache().count()

These filters will be applied to the object side reader before the KNN join is executed. 
If you intend to apply the filters after the KNN join, please ensure that you materialize the KNN join results before applying the filters. 
For example, you can use the following approach:

Scala Example:
val knnResult = knnJoinDF.cache()
val filteredResult = knnResult.filter(condition)

SQL Example:
CREATE OR REPLACE TEMP VIEW knnResult AS
SELECT * FROM (
  -- Your KNN join SQL here
) AS knnView
CACHE TABLE knnResult;
SELECT * FROM knnResult WHERE condition;
These filters will be applied to the object side reader before the KNN join is executed. 
If you intend to apply the filters after the KNN join, please ensure that you materialize the KNN join results before applying the filters. 
For example, you can use the following approach:

Scala Example:
val knnResult = knnJoinDF.cache()
val filteredResult = knnResult.filter(condition)

SQL Example:
CREATE OR REPLACE TEMP VIEW knnResult AS
SELECT * FR

2697

# Modelling

In [31]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder, VectorAssembler, MinMaxScaler
)
from pyspark.sql.functions import col

# Step 2: Convert booleans to integers
roads_feature_values.cache().count()

roads_feature_values_transformed = roads_feature_values_ranked.\
    withColumn("is_b", col("is_b").cast("int")).\
    withColumn("is_t", col("is_t").cast("int")).\
    withColumn("is_uc", col("is_uc").cast("int"))

def index_string_column(column_name: str) -> list:
    indexer = StringIndexer(
        inputCol=column_name,
        outputCol=f"{column_name}_index"
    )
    
    encoder = OneHotEncoder(
        inputCol=f"{column_name}_index",
        outputCol=f"{column_name}_encoded"
    )

    return [indexer, encoder]

def min_max_scale_column(column_name: str) -> list:
    ms_assembler = VectorAssembler(
        inputCols=[column_name],
        outputCol=f"{column_name}_vec"
    )
    
    ms_scaler = MinMaxScaler(
        inputCol=f"{column_name}_vec",
        outputCol=f"{column_name}_scaled"
    )

    return [ms_assembler, ms_scaler]

# Step 5: Convert label 'ac' to numeric
label_indexer = StringIndexer(
    inputCol="ac",
    outputCol="label"
)

# Step 6: Create vector 
assembler = VectorAssembler(
    inputCols=[
        "lc_encoded",
        "ms_scaled",
        "is_b",
        "is_t",
        "is_uc",
        "c",
        "distance_scaled",
        "promiles_scaled",
        "quality_encoded"
    ],
    outputCol="features"
)

# Step 7: Compose pipeline
pipeline = Pipeline(stages=[
    *index_string_column("lc"),
    *index_string_column("quality"),
    label_indexer,
    *min_max_scale_column("distance"),
    *min_max_scale_column("promiles"),
    *min_max_scale_column("ms"),
    assembler,
])

# Fit and transform
model = pipeline.fit(roads_feature_values_transformed)
data_for_modelling = model.transform(roads_feature_values_transformed)

These filters will be applied to the object side reader before the KNN join is executed. 
If you intend to apply the filters after the KNN join, please ensure that you materialize the KNN join results before applying the filters. 
For example, you can use the following approach:

Scala Example:
val knnResult = knnJoinDF.cache()
val filteredResult = knnResult.filter(condition)

SQL Example:
CREATE OR REPLACE TEMP VIEW knnResult AS
SELECT * FROM (
  -- Your KNN join SQL here
) AS knnView
CACHE TABLE knnResult;
SELECT * FROM knnResult WHERE condition;
These filters will be applied to the object side reader before the KNN join is executed. 
If you intend to apply the filters after the KNN join, please ensure that you materialize the KNN join results before applying the filters. 
For example, you can use the following approach:

Scala Example:
val knnResult = knnJoinDF.cache()
val filteredResult = knnResult.filter(condition)

SQL Example:
CREATE OR REPLACE TEMP VIEW knnResult AS
SELECT * FR

In [32]:
from xgboost.spark import SparkXGBClassifier

xgb_classifier = SparkXGBClassifier(
  features_col="features",
  label_col="label",
  num_workers=10,
)

In [33]:
train_df, test_df = data_for_modelling.randomSplit([0.8, 0.2], seed=42)

In [ ]:
xgboost_model = xgb_classifier.fit(train_df)

In [35]:
predictions = xgboost_model.transform(test_df)

In [36]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})
recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")

2025-06-19 00:23:18,916 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2025-06-19 00:23:25,986 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2025-06-19 00:23:33,292 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
[Stage 310:====================================================>(197 + 3) / 200]

Accuracy: 0.7505
F1 Score: 0.7519
Recall: 0.7505


# CrossValidation

In [47]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

paramGrid = ParamGridBuilder()\
  .addGrid(xgb_classifier.max_depth, [2, 3, 4, 5, 6, 7])\
  .addGrid(xgb_classifier.n_estimators, [50, 60, 70, 80, 90, 100])\
  .build()

In [48]:
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

In [49]:
cv = CrossValidator(
    estimator=xgb_classifier,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)

In [ ]:
xgboost_model_cv = cv.fit(train_df)
predictions = xgboost_model_cv.transform(test_df)

In [51]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})
recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Recall: {recall:.4f}")

2025-06-19 00:59:16,986 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2025-06-19 00:59:22,964 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
2025-06-19 00:59:28,254 INFO XGBoost-PySpark: predict_udf Do the inference on the CPUs
[Stage 8179:================================================>  (190 + 10) / 200]

Accuracy: 0.7352
F1 Score: 0.7370
Recall: 0.7352
